In [1]:
# %%
# ============================================================
# GLOBAL PARAMETER MAPS — phi, w (TMDSI) and p, q (scPDSI)
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.interpolate import griddata



In [2]:
# %%
# ============================================================
# GLOBAL PARAMETER MAPS — phi, w (TMDSI) and p, q (scPDSI)
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.interpolate import griddata

# ── interpolate to global regular grid ───────────────────────
def to_global_grid(lons, lats, vals, res_lon=720, res_lat=360):
    lo = np.linspace(-180, 180, res_lon)
    la = np.linspace(-90,   90, res_lat)
    LO, LA = np.meshgrid(lo, la)
    pts = np.column_stack([lons, lats])
    grd = griddata(pts, vals, (LO, LA), method='linear')
    return LO, LA, grd

# pre-compute all global grids (slow once, reused)
print("Interpolating to global grid...")
LO, LA, grd_p   = to_global_grid(lons, lats, sc_p)
LO, LA, grd_q   = to_global_grid(lons, lats, sc_q)

# phi and w: annual mean across 12 months
phi_mean = np.nanmean(tm_phi, axis=0)   # (78016,)
w_mean   = np.nanmean(tm_w,   axis=0)

LO, LA, grd_phi_mean = to_global_grid(lons, lats, phi_mean)
LO, LA, grd_w_mean   = to_global_grid(lons, lats, w_mean)

# seasonal phi: Jan(0), Apr(3), Jul(6), Oct(9)
seasonal_months = [0, 3, 6, 9]
season_labels   = ["January", "April", "July", "October"]
grd_phi_seas = []
grd_w_seas   = []
for mo in seasonal_months:
    _, _, g1 = to_global_grid(lons, lats, tm_phi[mo])
    _, _, g2 = to_global_grid(lons, lats, tm_w[mo])
    grd_phi_seas.append(g1)
    grd_w_seas.append(g2)

print("Done.")

# ── shared map drawing function ───────────────────────────────
def draw_map(ax, LO, LA, grid, cmap, norm,
             title="", unit="", cb_ax=None):
    ax.set_facecolor("#CADAED")
    pc = ax.pcolormesh(LO, LA, grid,
                       cmap=cmap, norm=norm,
                       transform=ccrs.PlateCarree(),
                       zorder=2, shading='auto',
                       rasterized=True)
    ax.add_feature(cfeature.OCEAN,
                   facecolor="#CADAED", zorder=3)
    ax.add_feature(cfeature.COASTLINE,
                   edgecolor="#333333", linewidth=0.4, zorder=4)
    ax.add_feature(cfeature.BORDERS,
                   edgecolor="#555555", linewidth=0.25, zorder=4)
    ax.set_global()
    ax.set_title(title, fontsize=9, pad=4,
                 fontweight="bold", color="#1a1a1a")
    gl = ax.gridlines(linewidth=0.25, color="#888888",
                      alpha=0.4, linestyle="--",
                      xlocs=range(-180,181,60),
                      ylocs=range(-90,91,30))
    gl.top_labels = False; gl.right_labels = False
    gl.bottom_labels = False; gl.left_labels = False
    return pc

proj = ccrs.Mercator(central_longitude=0,
                     min_latitude=-75, max_latitude=85)


Interpolating to global grid...


NameError: name 'lons' is not defined

In [3]:

# ================================================================
# FIGURE 1 — scPDSI p and q  (2 panels)
# ================================================================

fig1 = plt.figure(figsize=(18, 9), facecolor="#F2F0EC")
fig1.patch.set_facecolor("#F2F0EC")

gs1 = gridspec.GridSpec(2, 1, figure=fig1,
                         hspace=0.30,
                         left=0.03, right=0.97,
                         top=0.91, bottom=0.10)

# ── p colormap: 0.5–0.99 sequential (purple-ish)
cmap_p  = plt.cm.get_cmap("plasma_r")
norm_p  = mcolors.Normalize(vmin=0.50, vmax=0.99)

# ── q colormap: 0–0.5 sequential (teal)
cmap_q  = plt.cm.get_cmap("YlGnBu")
norm_q  = mcolors.Normalize(vmin=0.01, vmax=0.50)

ax_p = fig1.add_subplot(gs1[0], projection=proj)
ax_q = fig1.add_subplot(gs1[1], projection=proj)

pc_p = draw_map(ax_p, LO, LA, grd_p, cmap_p, norm_p,
                title="scPDSI Persistence Coefficient  p  "
                      "(larger = longer drought memory)")
pc_q = draw_map(ax_q, LO, LA, grd_q, cmap_q, norm_q,
                title="scPDSI Responsiveness Coefficient  q  "
                      "(larger = stronger response to current-month anomaly)")

# colorbars
for ax, pc, label in [
    (ax_p, pc_p, "p  (dimensionless)"),
    (ax_q, pc_q, "q  (dimensionless)")
]:
    pos  = ax.get_position()
    cpos = [pos.x0 + 0.25*(pos.x1-pos.x0),
            pos.y0 - 0.035,
            0.50*(pos.x1-pos.x0),
            0.012]
    cbax = fig1.add_axes(cpos)
    cb   = fig1.colorbar(pc, cax=cbax, orientation="horizontal")
    cb.set_label(label, fontsize=8, color="#333333")
    cb.ax.tick_params(labelsize=7)
    cb.outline.set_edgecolor("#aaaaaa")

fig1.suptitle(
    "Global scPDSI Calibration Parameters  ·  "
    "Calibration Period 1979–2010  ·  CPC 0.25°",
    fontsize=13, fontweight="bold", y=0.97, color="#111111"
)

plt.savefig("params_scpdsi_pq_global.png",
            dpi=200, bbox_inches="tight",
            facecolor=fig1.get_facecolor())
plt.show()
print("Saved: params_scpdsi_pq_global.png")


# ================================================================
# FIGURE 2 — TMDSI annual mean phi and w  (2 panels)
# ================================================================

fig2 = plt.figure(figsize=(18, 9), facecolor="#F2F0EC")
fig2.patch.set_facecolor("#F2F0EC")

gs2 = gridspec.GridSpec(2, 1, figure=fig2,
                         hspace=0.30,
                         left=0.03, right=0.97,
                         top=0.91, bottom=0.10)

cmap_phi = plt.cm.get_cmap("RdYlGn")      # red=low memory, green=high memory
norm_phi = mcolors.Normalize(vmin=0.10, vmax=0.99)

cmap_w   = plt.cm.get_cmap("RdYlGn_r")   # green=low w (high phi), red=high w
norm_w   = mcolors.Normalize(vmin=0.10, vmax=0.99)

ax_phi = fig2.add_subplot(gs2[0], projection=proj)
ax_w   = fig2.add_subplot(gs2[1], projection=proj)

pc_phi = draw_map(ax_phi, LO, LA, grd_phi_mean, cmap_phi, norm_phi,
                  title=r"TMDSI Annual-Mean Persistence  $\bar{\phi}_m$  "
                        r"(physics-derived from soil drainage)")
pc_w   = draw_map(ax_w,   LO, LA, grd_w_mean,   cmap_w,   norm_w,
                  title=r"TMDSI Annual-Mean Responsiveness  $\bar{w}_m = \sqrt{1-\phi_m^2}$")

for ax, pc, label in [
    (ax_phi, pc_phi, r"$\bar{\phi}$  (dimensionless)"),
    (ax_w,   pc_w,   r"$\bar{w}$  (dimensionless)")
]:
    pos  = ax.get_position()
    cpos = [pos.x0 + 0.25*(pos.x1-pos.x0),
            pos.y0 - 0.035,
            0.50*(pos.x1-pos.x0),
            0.012]
    cbax = fig2.add_axes(cpos)
    cb   = fig2.colorbar(pc, cax=cbax, orientation="horizontal")
    cb.set_label(label, fontsize=8, color="#333333")
    cb.ax.tick_params(labelsize=7)
    cb.outline.set_edgecolor("#aaaaaa")

fig2.suptitle(
    r"Global TMDSI Parameters $\phi_m$ and $w_m$  ·  "
    "Annual Mean  ·  CPC 0.25°",
    fontsize=13, fontweight="bold", y=0.97, color="#111111"
)

plt.savefig("params_tmdsi_phi_w_annual.png",
            dpi=200, bbox_inches="tight",
            facecolor=fig2.get_facecolor())
plt.show()
print("Saved: params_tmdsi_phi_w_annual.png")


# ================================================================
# FIGURE 3 — TMDSI seasonal phi  (2 rows × 4 months)
# top row = phi, bottom row = w
# ================================================================

fig3 = plt.figure(figsize=(20, 9), facecolor="#F2F0EC")
fig3.patch.set_facecolor("#F2F0EC")

gs3 = gridspec.GridSpec(2, 4, figure=fig3,
                         hspace=0.22, wspace=0.04,
                         left=0.02, right=0.98,
                         top=0.91, bottom=0.10)

for col, (slbl, gphi, gw) in enumerate(
        zip(season_labels, grd_phi_seas, grd_w_seas)):

    ax_top = fig3.add_subplot(gs3[0, col], projection=proj)
    ax_bot = fig3.add_subplot(gs3[1, col], projection=proj)

    draw_map(ax_top, LO, LA, gphi, cmap_phi, norm_phi, title=slbl)
    draw_map(ax_bot, LO, LA, gw,   cmap_w,   norm_w,   title=slbl)

# row labels
fig3.text(0.01, 0.72, r"$\phi_m$", fontsize=13, fontweight="bold",
          va="center", ha="left", color="#1a1a1a", rotation=90)
fig3.text(0.01, 0.30, r"$w_m$",    fontsize=13, fontweight="bold",
          va="center", ha="left", color="#1a1a1a", rotation=90)

# shared colorbars
for row_y, pc_ref, label in [
    (0.085, mcolors.Normalize(vmin=0.10, vmax=0.99),
     r"$\phi_m$ — Persistence  (dimensionless)"),
    (0.03,  mcolors.Normalize(vmin=0.10, vmax=0.99),
     r"$w_m$ — Responsiveness  (dimensionless)"),
]:
    cbax = fig3.add_axes([0.15, row_y, 0.70, 0.012])
    cmap_use = cmap_phi if "phi" in label else cmap_w
    sm = plt.cm.ScalarMappable(cmap=cmap_use, norm=pc_ref)
    sm.set_array([])
    cb = fig3.colorbar(sm, cax=cbax, orientation="horizontal")
    cb.set_label(label, fontsize=8, color="#333333")
    cb.ax.tick_params(labelsize=7)
    cb.outline.set_edgecolor("#aaaaaa")

fig3.suptitle(
    r"Seasonal Variation of TMDSI Parameters $\phi_m$ and $w_m$  ·  "
    "CPC 0.25°  ·  Calibration 1979–2010",
    fontsize=13, fontweight="bold", y=0.97, color="#111111"
)

plt.savefig("params_tmdsi_phi_w_seasonal.png",
            dpi=200, bbox_inches="tight",
            facecolor=fig3.get_facecolor())
plt.show()
print("Saved: params_tmdsi_phi_w_seasonal.png")

C:\Users\ktripat\AppData\Local\Temp\ipykernel_48228\3045365447.py:14: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap_p  = plt.cm.get_cmap("plasma_r")
C:\Users\ktripat\AppData\Local\Temp\ipykernel_48228\3045365447.py:18: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap_q  = plt.cm.get_cmap("YlGnBu")


NameError: name 'proj' is not defined

<Figure size 1800x900 with 0 Axes>